In [ ]:
import pandas as pd
import numpy as np
np.random.seed( 42 )
from tqdm import tqdm

In [ ]:
df_mordred = pd.read_csv(r'df_mordred descriptors.csv')

In [ ]:
df_descriptors = pd.read_csv(r'Lipinski and RDKit descriptors.csv')

In [ ]:
# Combine the data frames horizontally
combined_df = pd.concat([df_mordred, df_descriptors], axis=1)

# Print the combined data frame
combined_df

In [ ]:
non_numeric_columns = df.select_dtypes(exclude=[float, int]).columns
non_numeric_columns

In [ ]:
combined_df = df.drop(columns=non_numeric_columns)

###  Remove duplicate columns

In [ ]:
combined_df = combined_df.T.drop_duplicates().T
combined_df

In [ ]:
combined_df.describe()

### drop out highly correlated features

In [ ]:
# Calculate the correlation matrix
corr_matrix = combined_df.corr()

# Find columns with high correlation
high_corr_columns = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            colname = corr_matrix.columns[i]
            high_corr_columns.add(colname)

# Drop columns with high correlation
combined_df = combined_df.drop(columns=high_corr_columns)

### Scale Descriptors DataFrame

In [ ]:
from sklearn import preprocessing
scaler = preprocessing.MinMaxScaler(feature_range=(0,1))
df_scaled = scaler.fit_transform(combined_df)
df_scaled = pd.DataFrame(df_scaled, columns=combined_df.columns)
df_scaled

In [ ]:
df_scaled.isnull().sum().any()

In [ ]:
# Check which columns have null values
null_columns = df_scaled.columns[df_scaled.isnull().any()]

# Drop null columns from the DataFrame
df_scaled = df_scaled.drop(null_columns, axis=1)

In [ ]:
df_scaled.isnull().sum().any()

In [ ]:
df_scaled.shape

### Removing features with low variance

In [ ]:
variance = df_scaled.var(axis=0)
variance.sort_values(ascending=False, inplace=True)
variance

In [ ]:
print(variance.max())
print(variance.min())
print(variance.mean())

In [ ]:
#get low variance columns
low_variance_columns = variance[variance <= 0.02].index.to_list()

In [ ]:
#Drop low variance columns
selected_desc = combined_df.drop(columns=low_variance_columns, axis=1)
selected_desc

In [ ]:
selected_desc_columns = selected_desc.columns

In [ ]:
from sklearn import preprocessing
scaler = preprocessing.MinMaxScaler(feature_range=(0,1))
selected_desc = scaler.fit_transform(selected_desc)
selected_desc = pd.DataFrame(selected_desc, columns=selected_desc_columns)
selected_desc

In [ ]:
selected_desc.describe()

In [ ]:
selected_desc.shape

In [ ]:
selected_desc.to_csv('selected_descriptors.csv')